In [1]:
import numpy as np
import scipy.stats as stats
import os

np.set_printoptions(suppress=True, precision=4)

In [2]:
Ker = 2 # Cambiar a 2 para evaluar el modelo NO lineal

if Ker == 1:  # LINEAL
    MC_SVM    = np.array([96.00, 57.90, 99.05, 57.32, 75.20, 72.09, 87.57, 69.70, 90.91])
    OVA_SVM   = np.array([94.67, 61.47, 98.57, 60.69, 74.05, 49.55, 88.06, 74.39, 92.68])
    OVO_SVM   = np.array([98.00, 64.88, 98.57, 66.12, 74.30, 89.14, 89.30, 79.35, 95.58])
    OvA_PSVM  = np.array([93.33, 59.48, 99.52, 60.55, 72.90, 57.05, 86.39, 70.01, 92.68])
    OvO_PSVM  = np.array([98.00, 60.28, 98.63, 68.00, 74.42, 89.27, 88.74, 78.53, 96.02])
    OvA_CPSVM = np.array([96.00, 60.60, 99.11, 60.20, 75.68, 57.86, 81.82, 71.66, 93.33])
    OvO_CPSVM = np.array([98.00, 60.99, 98.63, 67.37, 75.62, 85.82, 83.52, 79.42, 96.41])

else:         # NO LINEAL
    MC_SVM    = np.array([97.33, 87.78, 99.05, 71.44, 75.92, 99.05, 88.80, 83.20, 98.33])
    OVA_SVM   = np.array([97.33, 87.22, 99.52, 71.76, 74.17, 99.62, 88.80, 84.37, 97.50])
    OVO_SVM   = np.array([98.00, 87.70, 99.05, 72.22, 74.65, 99.64, 90.68, 82.65, 97.40])
    OvA_PSVM  = np.array([98.67, 87.78, 100.00, 71.99, 74.70, 99.45, 89.57, 83.70, 97.58])
    OvO_PSVM  = np.array([98.67, 88.17, 99.52, 74.10, 74.48, 99.64, 90.68, 83.99, 97.92])
    OvA_CPSVM = np.array([97.33, 87.22, 100.00, 73.34, 75.65, 99.82, 89.91, 83.90, 97.84])
    OvO_CPSVM = np.array([98.67, 87.14, 100.00, 71.92, 75.48, 99.82, 89.91, 84.10, 98.14])

DAT = np.column_stack((
    MC_SVM, OVA_SVM, OVO_SVM, OvA_PSVM, OvO_PSVM, OvA_CPSVM, OvO_CPSVM
))

DATt = [
    'MC-SVM', 'OVA-SVM', 'OVO-SVM', 'OvA-PSVM', 'OvO-PSVM', 'OvA-CPSVM', 'OvO-CPSVM'
]

nd, nc = DAT.shape  # Número de datasets (nd) y clasificadores (nc)

In [3]:
rk = np.zeros_like(DAT)
for ii in range(nd):
    rk[ii, :] = stats.rankdata(-DAT[ii, :])

print("DAT Matrix:")
print(DAT)
print("\nRanking Matrix:")
print(rk)

# Rango Medio
Mrank = np.mean(rk, axis=0)
print("\nAverage Rank:")
print(Mrank)

# Definir el nombre del archivo según el tipo de kernel
filename_rank = "Ranking_Promedio_BACCU_lin.txt" if Ker == 1 else "Ranking_Promedio_BACCU_no_lin.txt"

# Crear el contenido uniendo cada modelo con su respectivo rango medio
rank_text = ""
for name, rank in zip(DATt, Mrank):
    rank_text += f"{name},{rank:.4f}\n"

# Guardar el archivo de texto
with open(filename_rank, "w", encoding="utf-8") as fid_rank:
    fid_rank.write(rank_text)
    
print(f"\n\nRankings promedio guardados exitosamente como '{filename_rank}'")

DAT Matrix:
[[ 97.33  97.33  98.    98.67  98.67  97.33  98.67]
 [ 87.78  87.22  87.7   87.78  88.17  87.22  87.14]
 [ 99.05  99.52  99.05 100.    99.52 100.   100.  ]
 [ 71.44  71.76  72.22  71.99  74.1   73.34  71.92]
 [ 75.92  74.17  74.65  74.7   74.48  75.65  75.48]
 [ 99.05  99.62  99.64  99.45  99.64  99.82  99.82]
 [ 88.8   88.8   90.68  89.57  90.68  89.91  89.91]
 [ 83.2   84.37  82.65  83.7   83.99  83.9   84.1 ]
 [ 98.33  97.5   97.4   97.58  97.92  97.84  98.14]]

Ranking Matrix:
[[6.  6.  4.  2.  2.  6.  2. ]
 [2.5 5.5 4.  2.5 1.  5.5 7. ]
 [6.5 4.5 6.5 2.  4.5 2.  2. ]
 [7.  6.  3.  4.  1.  2.  5. ]
 [1.  7.  5.  4.  6.  2.  3. ]
 [7.  5.  3.5 6.  3.5 1.5 1.5]
 [6.5 6.5 1.5 5.  1.5 3.5 3.5]
 [6.  1.  7.  5.  3.  4.  2. ]
 [1.  6.  7.  5.  3.  4.  2. ]]

Average Rank:
[4.8333 5.2778 4.6111 3.9444 2.8333 3.3889 3.1111]


Rankings promedio guardados exitosamente como 'Ranking_Promedio_BACCU_no_lin.txt'


In [4]:
print("\nTest:")

# Chi-cuadrado de Friedman
chi = (12 * nd / (nc * (nc + 1))) * (np.sum(Mrank**2) - (nc * (nc + 1)**2) / 4)
print(f"Chi-square (con {nc-1} grados de libertad): {chi:f}")

# Extensión Iman-Davenport
Friedman_test = (nd - 1) * chi / (nd * (nc - 1) - chi)
print(f"Friedman_test (F_F): {Friedman_test:f}")

# Valor crítico de F
alpha = 0.05
# stats.f.ppf es el equivalente de finv en MATLAB
df1 = nc - 1
df2 = (nc - 1) * (nd - 1)
F_crit = stats.f.ppf(1 - alpha, df1, df2)

print(f"\nValor crítico de F (con una significancia de {alpha:.2f}, {df1} y {df2} grados de libertad): {F_crit:f}\n")

print("Result Test")
print("-----------")
if F_crit > Friedman_test:
    print("H0 : No hay diferencia estadística entre los clasificadores en términos de BACCU.")
else:
    print("H1 : Existen diferencias estadísticas entre los clasificadores en términos de BACCU.")
    print("es decir, la hipótesis nula (H0) de la prueba de Friedman debe ser rechazada.")


Test:
Chi-square (con 6 grados de libertad): 10.083333
Friedman_test (F_F): 1.836812

Valor crítico de F (con una significancia de 0.05, 6 y 48 grados de libertad): 2.294601

Result Test
-----------
H0 : No hay diferencia estadística entre los clasificadores en términos de BACCU.


In [5]:
q_alpha = 2.949 # Para alpha=0.05 y nc=7 modelos, q_alpha es aprox 2.949 según la tabla de Nemenyi.
CD = np.sqrt((nc * (nc + 1)) / (6 * nd)) * q_alpha

print(f"\nPrueba de Nemenyi: CD= {CD:f}")

print("\nComparaciones uno a uno:")
for ii in range(nc):
    for jj in range(ii + 1, nc):
        diff = abs(Mrank[ii] - Mrank[jj])
        if diff > CD:
            print(f" {diff} > {CD}")
            print(f"H1 : Existen diferencias estadísticas entre {DATt[ii]} y {DATt[jj]}.\n")
        else:
            print(f" {diff} < {CD}")
            print(f"H0 : No hay diferencia estadística entre {DATt[ii]} y {DATt[jj]}.\n")


Prueba de Nemenyi: CD= 3.003115

Comparaciones uno a uno:
 0.44444444444444464 < 3.003114605427727
H0 : No hay diferencia estadística entre MC-SVM y OVA-SVM.

 0.22222222222222232 < 3.003114605427727
H0 : No hay diferencia estadística entre MC-SVM y OVO-SVM.

 0.8888888888888884 < 3.003114605427727
H0 : No hay diferencia estadística entre MC-SVM y OvA-PSVM.

 1.9999999999999996 < 3.003114605427727
H0 : No hay diferencia estadística entre MC-SVM y OvO-PSVM.

 1.4444444444444442 < 3.003114605427727
H0 : No hay diferencia estadística entre MC-SVM y OvA-CPSVM.

 1.7222222222222219 < 3.003114605427727
H0 : No hay diferencia estadística entre MC-SVM y OvO-CPSVM.

 0.666666666666667 < 3.003114605427727
H0 : No hay diferencia estadística entre OVA-SVM y OVO-SVM.

 1.333333333333333 < 3.003114605427727
H0 : No hay diferencia estadística entre OVA-SVM y OvA-PSVM.

 2.444444444444444 < 3.003114605427727
H0 : No hay diferencia estadística entre OVA-SVM y OvO-PSVM.

 1.8888888888888888 < 3.0031146

In [6]:
meanBACCU = np.nanmean(DAT, axis=0)
stdBACCU = np.nanstd(DAT, axis=0, ddof=1)
avgRank = np.nanmean(rk, axis=0)

# Orden ascendente por ranking
order = np.argsort(avgRank)

def escape_underscore(s):
    return s.replace("_", "\\_")

header = (
    "\\begin{table}[t]\n"
    "\\centering\n"
    "\\small\n"
    "\\caption{Resumen de desempeño promedio: media, desviación típica y rango medio del BACCU (mayor es mejor) sobre los seis conjuntos de datos.}\n"
    "\\label{tab:baccu_summary}\n"
    "\\begin{tabular}{lccc}\n"
    "\\toprule\n"
    "Modelo & $\\overline{\\mathrm{BACCU}}$ & SD & Average Rank \\\\\n"
    "\\midrule\n"
)

rows = ""
for j in order:
    name = escape_underscore(DATt[j])
    rows += f"{name} & {meanBACCU[j]:.2f} & {stdBACCU[j]:.2f} & {avgRank[j]:.2f} \\\\\n"

footer = (
    "\\bottomrule\n"
    "\\end{tabular}\n"
    "\\end{table}\n"
)

latexTable = header + rows + footer

print("\n--- LaTeX table (BACCU summary) ---")
print(latexTable)

# Guardar a archivo .tex
filename = "table_baccu_Multi.tex" if Ker == 1 else "table_baccu_Multi_Nonl.tex"
with open(filename, "w", encoding="utf-8") as fid:
    fid.write(latexTable)
    
print(f"Tabla guardada exitosamente como '{filename}'")


--- LaTeX table (BACCU summary) ---
\begin{table}[t]
\centering
\small
\caption{Resumen de desempeño promedio: media, desviación típica y rango medio del BACCU (mayor es mejor) sobre los seis conjuntos de datos.}
\label{tab:baccu_summary}
\begin{tabular}{lccc}
\toprule
Modelo & $\overline{\mathrm{BACCU}}$ & SD & Average Rank \\
\midrule
OvO-PSVM & 89.69 & 10.33 & 2.83 \\
OvO-CPSVM & 89.46 & 10.70 & 3.11 \\
OvA-CPSVM & 89.45 & 10.23 & 3.39 \\
OvA-PSVM & 89.27 & 10.73 & 3.94 \\
OVO-SVM & 89.11 & 10.59 & 4.61 \\
MC-SVM & 88.99 & 10.45 & 4.83 \\
OVA-SVM & 88.92 & 10.65 & 5.28 \\
\bottomrule
\end{tabular}
\end{table}

Tabla guardada exitosamente como 'table_baccu_Multi_Nonl.tex'
